# <h1 style="text-align: center;">1.DATA CLEANING</h1>


## 1. Cleaning demography table

## Reading csv file
We start by loading the raw demography data into a DataFrame so we can inspect and clean it.

In [1]:
import pandas as pd
import os
os.chdir("Python_Hackathon_SEP_2026")

df_demography=pd.read_csv("cardiac_failure/demography.csv")

## Renaming columns
Raw column names like inpatient_number and agecat aren't clear or consistent. We rename them to patient_id and age_category so the rest of the notebook (and anyone reading it later) can understand what each column actually means, and so column names match across all our tables.

In [2]:
# Mapping old column names to new column names
df_demography = df_demography.rename(columns={
    'inpatient_number': 'patient_id',
    'agecat': 'age_category'
})

In [3]:
df_demography

,patient_id,gender,weight,height,bmi,occupation,age_category
0,827040,Female,50.0,1.45,23.781213,NaN,69-79
1,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
2,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
3,866418,Male,70.0,1.70,24.221453,farmer,59-69
4,775928,Male,65.0,1.70,22.491349,UrbanResident,69-79
...,...,...,...,...,...,...,...
2003,740689,Female,35.0,1.50,15.555556,Others,79-89
2004,734280,Female,50.0,1.55,20.811655,UrbanResident,79-89
2005,781004,Male,75.0,1.70,25.951557,UrbanResident,39-49
2006,744870,Male,40.0,1.50,17.777778,UrbanResident,49-59


## Checking and Standardizing column values

Text columns often contain inconsistent formatting — extra spaces, mismatched capitalization, or missing entries — that can make the same real-world value look like different categories to pandas (e.g. `'male'` vs `'Male'`). We first fill missing `occupation` values with `'Unknown'` so we don't lose those rows, then strip whitespace and standardize capitalization for `gender`, `occupation`, and `age_category`, so identical values are always treated as the same category later on.

In [4]:
# 1. Fill NaNs first
df_demography['occupation'] = df_demography['occupation'].fillna('Unknown')

# 2. Clean text directly on the object series
df_demography['gender'] = df_demography['gender'].str.strip().str.title()
df_demography['occupation'] = df_demography['occupation'].str.strip().str.title()
df_demography['age_category'] = df_demography['age_category'].str.strip()



## Check for data types
We convert gender, occupation, and age_category to the category dtype, which is more memory-efficient and semantically correct for columns with a fixed set of repeating values.

In [5]:
df_demography.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    2008 non-null   int64  
 1   gender        2008 non-null   object 
 2   weight        2008 non-null   float64
 3   height        2008 non-null   float64
 4   bmi           2008 non-null   float64
 5   occupation    2008 non-null   object 
 6   age_category  2008 non-null   object 
dtypes: float64(3), int64(1), object(3)
memory usage: 109.9+ KB


In [6]:

df_demography['gender'] = df_demography['gender'].astype('category')
df_demography['occupation'] = df_demography['occupation'].astype('category')
df_demography['age_category'] = df_demography['age_category'].astype('category')

In [7]:
df_demography.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   patient_id    2008 non-null   int64   
 1   gender        2008 non-null   category
 2   weight        2008 non-null   float64 
 3   height        2008 non-null   float64 
 4   bmi           2008 non-null   float64 
 5   occupation    2008 non-null   category
 6   age_category  2008 non-null   category
dtypes: category(3), float64(3), int64(1)
memory usage: 69.5 KB


In [8]:
# Check unique values
print(df_demography['gender'].unique())
print(df_demography['occupation'].unique())
print(df_demography['age_category'].unique())

# Expected ideal output: array(['Female', 'Male'], dtype=object)
# If you see NaN, 'female', ' Male', or 'other', you'll need to clean it.

['Female', 'Male']
Categories (2, object): ['Female', 'Male']
['Unknown', 'Urbanresident', 'Farmer', 'Worker', 'Others', 'Officer']
Categories (6, object): ['Farmer', 'Officer', 'Others', 'Unknown', 'Urbanresident', 'Worker']
['69-79', '59-69', '79-89', '49-59', '89-110', '29-39', '39-49', '21-29']
Categories (8, object): ['21-29', '29-39', '39-49', '49-59', '59-69', '69-79', '79-89', '89-110']


In [9]:
df_demography

,patient_id,gender,weight,height,bmi,occupation,age_category
0,827040,Female,50.0,1.45,23.781213,Unknown,69-79
1,857781,Male,50.0,1.64,18.590125,Urbanresident,69-79
2,743087,Female,51.0,1.63,19.195303,Urbanresident,69-79
3,866418,Male,70.0,1.70,24.221453,Farmer,59-69
4,775928,Male,65.0,1.70,22.491349,Urbanresident,69-79
...,...,...,...,...,...,...,...
2003,740689,Female,35.0,1.50,15.555556,Others,79-89
2004,734280,Female,50.0,1.55,20.811655,Urbanresident,79-89
2005,781004,Male,75.0,1.70,25.951557,Urbanresident,39-49
2006,744870,Male,40.0,1.50,17.777778,Urbanresident,49-59


## Check for duplicate records

Duplicate rows can silently inflate patient counts and bias any analysis or model built on this data. We count exact duplicate rows and inspect them, so we can decide whether they're true duplicates (safe to drop) or coincidentally identical valid records.

In [10]:
# Count how many total exact duplicate rows exist
num_duplicates = df_demography.duplicated().sum()
print("Total exact duplicate rows:", num_duplicates)

# Display the duplicate rows if any exist
if num_duplicates > 0:
    print("\nDuplicate Rows:")
    print(df_demography[df_demography.duplicated(keep=False)])
else:
    print("No exact duplicate rows found.")

Total exact duplicate rows: 0
No exact duplicate rows found.


In [11]:
df_demography

,patient_id,gender,weight,height,bmi,occupation,age_category
0,827040,Female,50.0,1.45,23.781213,Unknown,69-79
1,857781,Male,50.0,1.64,18.590125,Urbanresident,69-79
2,743087,Female,51.0,1.63,19.195303,Urbanresident,69-79
3,866418,Male,70.0,1.70,24.221453,Farmer,59-69
4,775928,Male,65.0,1.70,22.491349,Urbanresident,69-79
...,...,...,...,...,...,...,...
2003,740689,Female,35.0,1.50,15.555556,Others,79-89
2004,734280,Female,50.0,1.55,20.811655,Urbanresident,79-89
2005,781004,Male,75.0,1.70,25.951557,Urbanresident,39-49
2006,744870,Male,40.0,1.50,17.777778,Urbanresident,49-59


## Checking for Outliers
Clinical measurements should fall within medically plausible ranges — for example, no adult weighs 0 kg or is 0.35 m tall. We compare each value against standard adult clinical reference ranges and flag anything outside them, because these are very likely data entry errors (wrong unit, missing digit, placeholder value) rather than real measurements. We also cross-check recorded BMI against BMI calculated from weight and height — a big mismatch signals a possible error in one of the underlying values even if each individual value looks "in range" on its own.

In [12]:
import pandas as pd

# Dynamically find the patient ID column name
id_col_matches = [c for c in df_demography.columns if any(k in c.lower() for k in ['patient', 'inpatient', 'id'])]
id_col = id_col_matches[0] if id_col_matches else df_demography.columns[0]

print(f"Detected patient ID column: '{id_col}'")

# Format the ID column cleanly as a string
df_demography[id_col] = (
    df_demography[id_col]
    .fillna('Unknown')
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

# Standard adult clinical reference ranges
CLINICAL_RANGES = {
    'weight': {'min': 25.0, 'max': 250.0, 'unit': 'kg'},
    'height': {'min': 1.20, 'max': 2.20,  'unit': 'm'},
    'bmi':    {'min': 12.0, 'max': 60.0,  'unit': 'kg/m²'}
}

outlier_records = []

# 1. Range Check (Weight, Height, BMI)
for col, bounds in CLINICAL_RANGES.items():
    if col in df_demography.columns:
        # Flag values strictly outside plausible clinical boundaries
        mask = (df_demography[col] < bounds['min']) | (df_demography[col] > bounds['max'])
        
        for idx in df_demography[mask].index:
            outlier_records.append({
                'patient_id': df_demography.loc[idx, id_col],
                'variable': col,
                'outlier_value': df_demography.loc[idx, col],
                'issue': f"Outside clinical range ({bounds['min']}–{bounds['max']} {bounds['unit']})"
            })

# 2. Formula Consistency Check (Calculated BMI vs Recorded BMI)
if all(col in df_demography.columns for col in ['weight', 'height', 'bmi']):
    valid_rows = df_demography.dropna(subset=['weight', 'height', 'bmi']).copy()
    calc_bmi = valid_rows['weight'] / (valid_rows['height'] ** 2)
    
    # Flag discrepancies larger than 1.0 unit
    mismatch_mask = abs(valid_rows['bmi'] - calc_bmi) > 1.0
    
    for idx in valid_rows[mismatch_mask].index:
        outlier_records.append({
            'patient_id': valid_rows.loc[idx, id_col],
            'variable': 'bmi_mismatch',
            'outlier_value': valid_rows.loc[idx, 'bmi'],
            'issue': f"Recorded BMI differs from weight/height² calculation ({calc_bmi.loc[idx]:.2f})"
        })

# 3. Output Outlier Summary
outliers_df = pd.DataFrame(outlier_records)

if not outliers_df.empty:
    print(f"\n=== DETECTED {len(outliers_df)} CLINICAL OUTLIERS ===")
    print(outliers_df.to_string(index=False))
else:
    print("\nNo clinical outliers or formula mismatches detected.")

Detected patient ID column: 'patient_id'

=== DETECTED 16 CLINICAL OUTLIERS ===
patient_id variable  outlier_value                                    issue
    730511   weight       0.000000   Outside clinical range (25.0–250.0 kg)
    785878   weight       0.000000   Outside clinical range (25.0–250.0 kg)
    775572   weight       0.000000   Outside clinical range (25.0–250.0 kg)
    756055   weight       8.000000   Outside clinical range (25.0–250.0 kg)
    837041   height       0.480000       Outside clinical range (1.2–2.2 m)
    815731   height       0.350000       Outside clinical range (1.2–2.2 m)
    805044   height       0.600000       Outside clinical range (1.2–2.2 m)
    844739   height       0.350000       Outside clinical range (1.2–2.2 m)
    837041      bmi     212.673611 Outside clinical range (12.0–60.0 kg/m²)
    815731      bmi     404.081633 Outside clinical range (12.0–60.0 kg/m²)
    805044      bmi     138.888889 Outside clinical range (12.0–60.0 kg/m²)
    7305

In [13]:
df_demography

,patient_id,gender,weight,height,bmi,occupation,age_category
0,827040,Female,50.0,1.45,23.781213,Unknown,69-79
1,857781,Male,50.0,1.64,18.590125,Urbanresident,69-79
2,743087,Female,51.0,1.63,19.195303,Urbanresident,69-79
3,866418,Male,70.0,1.70,24.221453,Farmer,59-69
4,775928,Male,65.0,1.70,22.491349,Urbanresident,69-79
...,...,...,...,...,...,...,...
2003,740689,Female,35.0,1.50,15.555556,Others,79-89
2004,734280,Female,50.0,1.55,20.811655,Urbanresident,79-89
2005,781004,Male,75.0,1.70,25.951557,Urbanresident,39-49
2006,744870,Male,40.0,1.50,17.777778,Urbanresident,49-59


## Imputing outliers for height, weight and bmi
Since we can't know what the true value should have been, we don't guess or fabricate a replacement — we set implausible weight/height values to NaN (missing) instead of leaving in physically impossible numbers. Because BMI is calculated from weight and height rather than independently measured, we recompute it fresh from the corrected weight/height columns.

In [14]:
import pandas as pd
import numpy as np

WEIGHT_RANGE = (25.0, 250.0)   # kg
HEIGHT_RANGE = (1.2, 2.2)      # m

# --- Step 1: Null out impossible weight/height values ---
df_demography.loc[~df_demography['weight'].between(*WEIGHT_RANGE), 'weight'] = np.nan
df_demography.loc[~df_demography['height'].between(*HEIGHT_RANGE), 'height'] = np.nan

# --- Step 2: Recompute BMI only where both weight & height are valid ---
df_demography['bmi'] = np.where(
    df_demography['weight'].notna() & df_demography['height'].notna(),
    df_demography['weight'] / (df_demography['height'] ** 2),
    np.nan
)

# --- Step 3: Sanity check ---
print(df_demography.loc[df_demography['patient_id'].isin(
    [730511, 785878, 775572, 756055, 837041, 815731, 805044, 844739]
), ['patient_id', 'weight', 'height', 'bmi']])

Empty DataFrame
Columns: [patient_id, weight, height, bmi]
Index: []


## Checking for Missing values

We check every column for missing values to understand exactly where the data is incomplete. This tells us which columns need a decision — fill with a sensible value, leave as missing because the gap is meaningful, or investigate further — rather than assuming the dataset is complete.

In [15]:
df_demography.isna().sum()

patient_id      0
gender          0
weight          4
height          4
bmi             8
occupation      0
age_category    0
dtype: int64

## Removing patient id 5 record since there are no records for this patient in other tables so removed the record
Before deleting a record, we check whether that patient appears in any of the other clinical tables (hospitalization, labs, patienthistory, etc.). Since patient 5 has no data anywhere else in the dataset, keeping their row in demography alone wouldn't be usable for any real analysis — an isolated record with no supporting clinical data. So we remove it from demography.csv

In [16]:
import os
import pandas as pd

folder_path = 'cardiac_failure'
target_patient = 5
demography_file = 'demography.csv'

# 1. Get all CSV files in the directory
all_csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
other_csv_files = [f for f in all_csv_files if f != demography_file]

found_in_other_tables = False

print(f"--- CHECKING PATIENT {target_patient} ACROSS OTHER CSV FILES ---")

for file_name in other_csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    id_cols = [c for c in df.columns if 'patient_id' in c.lower() or c.lower() == 'id']
    if id_cols:
        col_name = id_cols[0]
        matches = df[(df[col_name] == target_patient) | (df[col_name] == str(target_patient))]
        if len(matches) > 0:
            found_in_other_tables = True
            print(f"[FOUND] Patient {target_patient} exists in '{file_name}' ({len(matches)} record(s)).")

# 2. Use your already-cleaned df_demography — do NOT reload from disk here
if not found_in_other_tables:
    print(f"\n[NO MATCH] Patient {target_patient} was NOT found in any other CSV files.")

    id_col = 'patient_id'  # you already renamed this earlier in your cleaning steps
    print(f"Using column '{id_col}' for filtering in '{demography_file}'.")

    initial_count = len(df_demography)
    df_demography = df_demography[
        (df_demography[id_col] != target_patient) & 
        (df_demography[id_col] != str(target_patient))
    ].reset_index(drop=True)   # <-- assign back to df_demography, not a new variable

    print(f"[REMOVED] Patient {target_patient} removed from '{demography_file}'.")
    print(f"Row count: {initial_count} -> {len(df_demography)}")
else:
    print(f"\n[RETAINED] Patient {target_patient} kept in demography because records were found elsewhere.")
    
   
  

--- CHECKING PATIENT 5 ACROSS OTHER CSV FILES ---

[NO MATCH] Patient 5 was NOT found in any other CSV files.
Using column 'patient_id' for filtering in 'demography.csv'.
[REMOVED] Patient 5 removed from 'demography.csv'.
Row count: 2008 -> 2008


In [17]:
df_demography

,patient_id,gender,weight,height,bmi,occupation,age_category
0,827040,Female,50.0,1.45,23.781213,Unknown,69-79
1,857781,Male,50.0,1.64,18.590125,Urbanresident,69-79
2,743087,Female,51.0,1.63,19.195303,Urbanresident,69-79
3,866418,Male,70.0,1.70,24.221453,Farmer,59-69
4,775928,Male,65.0,1.70,22.491349,Urbanresident,69-79
...,...,...,...,...,...,...,...
2003,740689,Female,35.0,1.50,15.555556,Others,79-89
2004,734280,Female,50.0,1.55,20.811655,Urbanresident,79-89
2005,781004,Male,75.0,1.70,25.951557,Urbanresident,39-49
2006,744870,Male,40.0,1.50,17.777778,Urbanresident,49-59


## 2. Cleaning hospitalization table

In [18]:
df_hospitalization=pd.read_csv("cardiac_failure/hospitalization_discharge.csv")

In [19]:
df_hospitalization.head(5)

,inpatient_number,destinationdischarge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,dischargeday,admission_date,...,death_within_28_days,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02 00:00:00,...,0,1,0,1,0,1,NaN,19.0,1.0,19.0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN


In [20]:
df_hospitalization.columns.tolist()

['inpatient_number',
 'destinationdischarge',
 'admission_ward',
 'admission_way',
 'discharge_department',
 'visit_times',
 'respiratory_support',
 'oxygen_inhalation',
 'dischargeday',
 'admission_date',
 'outcome_during_hospitalization',
 'death_within_28_days',
 're_admission_within_28_days',
 'death_within_3_months',
 're_admission_within_3_months',
 'death_within_6_months',
 're_admission_within_6_months',
 'time_of_death__days_from_admission',
 'readmission_time_days_from_admission',
 'return_to_emergency_department_within_6_months',
 'time_to_emergency_department_within_6_months']

## Renaming columns

Like the demography table, the raw column names here are inconsistent — some are vague (`destinationdischarge`), some are verbose (`re_admission_within_28_days`), and none match the naming used in our other tables. We rename them to shorter, consistent names so the columns are easier to reference in code and so key columns like `patient_id` line up across all tables for merging later.

In [21]:
df_hospitalization = df_hospitalization.rename(columns={

    'inpatient_number': 'patient_id',

    # Admission / discharge details
    
    
    'admission_way': 'admission_type',
    
    'destinationdischarge': 'discharge_destination',
    'dischargeday': 'discharge_day',
    'visit_times': 'visit_count',

    # Clinical support during stay
    
    'oxygen_inhalation': 'oxygen_therapy',
    'outcome_during_hospitalization': 'hospitalization_outcome',

    # 28-day outcomes
    'death_within_28_days': 'death_28d',
    're_admission_within_28_days': 'readmission_28d',

    # 3-month outcomes
    'death_within_3_months': 'death_3m',
    're_admission_within_3_months': 'readmission_3m',

    # 6-month outcomes
    'death_within_6_months': 'death_6m',
    're_admission_within_6_months': 'readmission_6m',
    'return_to_emergency_department_within_6_months': 'emergency_return_6m',
    'time_to_emergency_department_within_6_months': 'emergency_return_time_days',

    # Time-to-event fields
    'time_of_death__days_from_admission': 'death_time_days',
    'readmission_time_days_from_admission': 'readmission_time_days',
})

print(df_hospitalization.columns.tolist())

['patient_id', 'discharge_destination', 'admission_ward', 'admission_type', 'discharge_department', 'visit_count', 'respiratory_support', 'oxygen_therapy', 'discharge_day', 'admission_date', 'hospitalization_outcome', 'death_28d', 'readmission_28d', 'death_3m', 'readmission_3m', 'death_6m', 'readmission_6m', 'death_time_days', 'readmission_time_days', 'emergency_return_6m', 'emergency_return_time_days']


## Checking for data types

Before doing any calculations or comparisons, we check how pandas has inferred each column's type. This helps us catch columns that should be numeric, datetime, or categorical but were loaded as generic text (`object`), so we know exactly what needs converting before we start analyzing the data.

In [22]:
df_hospitalization.dtypes

patient_id                      int64
discharge_destination          object
admission_ward                 object
admission_type                 object
discharge_department           object
visit_count                     int64
respiratory_support            object
oxygen_therapy                 object
discharge_day                   int64
admission_date                 object
hospitalization_outcome        object
death_28d                       int64
readmission_28d                 int64
death_3m                        int64
readmission_3m                  int64
death_6m                        int64
readmission_6m                  int64
death_time_days               float64
readmission_time_days         float64
emergency_return_6m           float64
emergency_return_time_days    float64
dtype: object

In [23]:
df_hospitalization.head(5)

,patient_id,discharge_destination,admission_ward,admission_type,discharge_department,visit_count,respiratory_support,oxygen_therapy,discharge_day,admission_date,...,death_28d,readmission_28d,death_3m,readmission_3m,death_6m,readmission_6m,death_time_days,readmission_time_days,emergency_return_6m,emergency_return_time_days
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02 00:00:00,...,0,1,0,1,0,1,NaN,19.0,1.0,19.0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN


In [24]:
import pandas as pd

# Step 1: Convert admission_date to datetime
df_hospitalization['admission_date'] = pd.to_datetime(
    df_hospitalization['admission_date'], errors='coerce'
)
print("Unparseable admission_date values:", df_hospitalization['admission_date'].isna().sum())

# step2 converting emergency_return_6m to Int64
df_hospitalization['emergency_return_6m'] = df_hospitalization['emergency_return_6m'].astype('Int64')


# Step 3: Convert all remaining object dtype columns to category
# (excludes admission_date since it's now datetime, and any non-object columns)
object_cols = df_hospitalization.select_dtypes(include='object').columns.tolist()
df_hospitalization[object_cols] = df_hospitalization[object_cols].astype('category')

# Step 4: Confirm final dtypes
print(df_hospitalization.dtypes)

Unparseable admission_date values: 0
patient_id                             int64
discharge_destination               category
admission_ward                      category
admission_type                      category
discharge_department                category
visit_count                            int64
respiratory_support                 category
oxygen_therapy                      category
discharge_day                          int64
admission_date                datetime64[ns]
hospitalization_outcome             category
death_28d                              int64
readmission_28d                        int64
death_3m                               int64
readmission_3m                         int64
death_6m                               int64
readmission_6m                         int64
death_time_days                      float64
readmission_time_days                float64
emergency_return_6m                    Int64
emergency_return_time_days           float64
dtype: object


## Duplicated check

Just as with demography, duplicate rows here would double-count patient outcomes (deaths, readmissions, stays) and distort any statistics computed from this table. We check for exact duplicate rows before proceeding with the rest of the cleaning.

In [25]:
# Count total exact duplicate rows
duplicate_count = df_hospitalization.duplicated().sum()
print(f"Total fully duplicate rows: {duplicate_count}")



Total fully duplicate rows: 0


## Checking for columns standardization

We inspect the unique values in each categorical column to check for inconsistent spellings, capitalization, or extra whitespace that would cause pandas to treat the same real-world category (e.g. two spellings of the same ward name) as multiple different ones.

In [26]:
for col in ['discharge_destination','admission_ward','admission_type',
            'discharge_department','oxygen_therapy','hospitalization_outcome']:
    print(col, '->', df_hospitalization[col].unique())

discharge_destination -> ['Home', 'HealthcareFacility', 'Unknown', 'Died']
Categories (4, object): ['Died', 'HealthcareFacility', 'Home', 'Unknown']
admission_ward -> ['Cardiology', 'GeneralWard', 'ICU', 'Others']
Categories (4, object): ['Cardiology', 'GeneralWard', 'ICU', 'Others']
admission_type -> ['NonEmergency', 'Emergency']
Categories (2, object): ['Emergency', 'NonEmergency']
discharge_department -> ['Cardiology', 'Others', 'GeneralWard', 'ICU']
Categories (4, object): ['Cardiology', 'GeneralWard', 'ICU', 'Others']
oxygen_therapy -> ['OxygenTherapy', 'AmbientAir']
Categories (2, object): ['AmbientAir', 'OxygenTherapy']
hospitalization_outcome -> ['Alive', 'Dead', 'DischargeAgainstOrder']
Categories (3, object): ['Alive', 'Dead', 'DischargeAgainstOrder']


## Checking for Outliers
We tried IQR (the standard outlier-detection method) first, but it didn't work well for this data. Most patients here have short stays and only 1 visit, so the "normal range" IQR calculated was extremely narrow — it ended up flagging things like a 2nd hospital visit as an "outlier," even though that's completely normal for real patients. It even gave a negative number as a valid lower limit for days, which doesn't make sense — you can't have negative days.

So instead of using a statistical formula, we just checked the values using common sense / medical logic:

discharge_day and visit_count should never be 0 or negative — every patient has at least 1 visit and stays at least 1 day.
If a patient is marked as having died within 6 months, their recorded day of death should also fall within that same 6-month window — otherwise the two columns are contradicting each other.

This way, we only flag values that are actually impossible or contradictory, instead of flagging normal (but less common) patients as errors.
Both checks came back with 0 problems, so this data is clean — no fixes needed.

In [27]:
# discharge_day / visit_times should never be 0 or negative
print("discharge_day <= 0:", (df_hospitalization['discharge_day'] <= 0).sum())
print("visit_times <= 0:", (df_hospitalization['visit_count'] <= 0).sum())

# time_of_death should never exceed the stated outcome window it's tied to
# e.g., if death_within_6_months=1, time_of_death should be <= ~183 days
inconsistent_death_time = df_hospitalization[
    (df_hospitalization['death_6m'] == 1) & 
    (df_hospitalization['death_time_days'] > 183)
]
print("death_6m=1 but time_of_death > 183 days:", len(inconsistent_death_time))

discharge_day <= 0: 0
visit_times <= 0: 0
death_6m=1 but time_of_death > 183 days: 0


In [28]:
def find_iqr_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: valid range ~[{lower:.1f}, {upper:.1f}] -> {len(outliers)} outliers")
    return outliers

numeric_cols = ['visit_count', 'discharge_day', 'death_time_days', 
                 'readmission_time_days', 'emergency_return_time_days']

for col in numeric_cols:
    find_iqr_outliers(df_hospitalization, col)

visit_count: valid range ~[1.0, 1.0] -> 148 outliers
discharge_day: valid range ~[0.0, 16.0] -> 163 outliers
death_time_days: valid range ~[-23.1, 43.9] -> 5 outliers
readmission_time_days: valid range ~[-144.0, 352.0] -> 55 outliers
emergency_return_time_days: valid range ~[-144.0, 352.0] -> 55 outliers


## Checking for Missing values
Most of the missing values in this table are not data quality problems — they follow a clear, explainable pattern. death_time_days, readmission_time_days, and emergency_return_time_days are time-to-event columns that are only populated when the corresponding event actually occurred, so their missing values simply mean "this did not happen" rather than "this is unknown." We leave these as NaN to preserve that meaning accurately, since filling them with 0 or any placeholder would incorrectly imply the event happened immediately.

For respiratory_support, although the ~98% missing rate suggests it likely means "no respiratory support needed," we cannot confirm this from the data dictionary, so we leave it as NaN rather than asserting something we can't verify.

In [29]:
# Total missing values per column
missing_counts = df_hospitalization.isna().sum()
print(missing_counts)

patient_id                       0
discharge_destination            0
admission_ward                   0
admission_type                   0
discharge_department             0
visit_count                      0
respiratory_support           1966
oxygen_therapy                   0
discharge_day                    0
admission_date                   0
hospitalization_outcome          0
death_28d                        0
readmission_28d                  0
death_3m                         0
readmission_3m                   0
death_6m                         0
readmission_6m                   0
death_time_days               1964
readmission_time_days         1107
emergency_return_6m              1
emergency_return_time_days    1111
dtype: int64


## 3.Cleaning patienthistory table

## Reading csv file

We start by loading the raw patient history (comorbidity) data into a DataFrame so we can inspect and clean it.

In [30]:
df_patienthistory=pd.read_csv("cardiac_failure/patienthistory.csv")

In [31]:
df_patienthistory.head(5)

,inpatient_number,cerebrovascular_disease,dementia,chronic_obstructive_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,diabetes,moderate_to_severe_chronic_kidney_disease,hemiplegia,leukemia,malignant_lymphoma,solid_tumor,liver_disease,aids,cci_score,type_ii_respiratory_failure,acute_renal_failure
0,857781,0,0,1,0,0.0,1,0.0,0,0,0,0,0.0,0,2.0,nontypeii,0
1,743087,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
2,866418,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
3,775928,0,0,1,0,0.0,0,1.0,0,0,0,0,0.0,0,2.0,nontypeii,0
4,810128,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0


## Renaming column names

As with the other tables, we rename `inpatient_number` to `patient_id` so the key column matches across all tables for merging later. We also rename `type_ii_respiratory_failure` to `type2_respiratory_failure` for cleaner, more consistent naming.

## Checking and Standardizing column values

Every other comorbidity column in this table is a clean binary flag (0/1), but `type2_respiratory_failure` was recorded as text (`'typeii'` / `'nontypeii'`). We map these text values to `1` and `0` so this column matches the binary pattern used everywhere else in the table, which makes it directly usable in calculations (e.g. `sum()`, `mean()`) and models without special handling.

In [32]:
import pandas as pd

df_patienthistory = pd.read_csv("cardiac_failure/patienthistory.csv")

# Now run the full corrected block fresh, top to bottom
df_patienthistory = df_patienthistory.rename(columns={
    'inpatient_number': 'patient_id',
    'type_ii_respiratory_failure': 'type2_respiratory_failure'
})

print(df_patienthistory['type2_respiratory_failure'].unique())  # confirm raw values before mapping

df_patienthistory['type2_respiratory_failure'] = df_patienthistory['type2_respiratory_failure'].map({
    'typeii': 1,
    'nontypeii': 0
})
print("Unmapped/missing after value standardization:", df_patienthistory['type2_respiratory_failure'].isna().sum())

['nontypeii' 'typeii']
Unmapped/missing after value standardization: 0


## Checking for data types

We check how pandas has inferred each column's type. Several comorbidity flag columns (`peptic_ulcer_disease`, `moderate_to_severe_chronic_kidney_disease`, `liver_disease`, `cci_score`) show up as `float64` instead of whole numbers — this happens because each has a few missing values, and standard `int64` cannot represent NaN. We convert these, along with the now-numeric `type2_respiratory_failure`, to pandas' nullable `Int64` type instead, which keeps them as clean whole numbers while still correctly preserving the missing values, rather than leaving them as misleading decimals.

In [33]:
df_patienthistory.dtypes

patient_id                                     int64
cerebrovascular_disease                        int64
dementia                                       int64
chronic_obstructive_pulmonary_disease          int64
connective_tissue_disease                      int64
peptic_ulcer_disease                         float64
diabetes                                       int64
moderate_to_severe_chronic_kidney_disease    float64
hemiplegia                                     int64
leukemia                                       int64
malignant_lymphoma                             int64
solid_tumor                                    int64
liver_disease                                float64
aids                                           int64
cci_score                                    float64
type2_respiratory_failure                      int64
acute_renal_failure                            int64
dtype: object

In [34]:
nullable_int_cols = ['peptic_ulcer_disease', 'moderate_to_severe_chronic_kidney_disease', 
                      'liver_disease', 'cci_score']

df_patienthistory[nullable_int_cols] = df_patienthistory[nullable_int_cols].astype('Int64')

# Confirm
print(df_patienthistory.dtypes)

patient_id                                   int64
cerebrovascular_disease                      int64
dementia                                     int64
chronic_obstructive_pulmonary_disease        int64
connective_tissue_disease                    int64
peptic_ulcer_disease                         Int64
diabetes                                     int64
moderate_to_severe_chronic_kidney_disease    Int64
hemiplegia                                   int64
leukemia                                     int64
malignant_lymphoma                           int64
solid_tumor                                  int64
liver_disease                                Int64
aids                                         int64
cci_score                                    Int64
type2_respiratory_failure                    int64
acute_renal_failure                          int64
dtype: object


## Check for duplicate records

Duplicate rows would double-count patients' comorbidity profiles and distort any statistics or models built from this table. We check for exact duplicate rows before proceeding.

In [35]:
# Count total exact duplicate rows
duplicate_count = df_patienthistory.duplicated().sum()
print("Total fully duplicate rows:", duplicate_count)

Total fully duplicate rows: 0


## Checking for Outliers

Since almost every column here is a binary comorbidity flag rather than a continuous measurement, the IQR method used for numeric tables doesn't apply — a flag is either valid (0, 1, or missing) or invalid, with no meaningful concept of a statistical outlier. Instead, we use domain logic: confirm every flag column only contains `0`, `1`, or a missing value, and confirm `cci_score` (the Charlson Comorbidity Index) falls within its valid clinical range.

In [36]:
binary_cols = ['cerebrovascular_disease', 'dementia', 'chronic_obstructive_pulmonary_disease',
               'connective_tissue_disease', 'peptic_ulcer_disease', 'diabetes',
               'moderate_to_severe_chronic_kidney_disease', 'hemiplegia', 'leukemia',
               'malignant_lymphoma', 'solid_tumor', 'liver_disease', 'aids',
               'type2_respiratory_failure', 'acute_renal_failure']

for col in binary_cols:
    print(col, '->', df_patienthistory[col].unique())

cerebrovascular_disease -> [0 1]
dementia -> [0 1]
chronic_obstructive_pulmonary_disease -> [1 0]
connective_tissue_disease -> [0 1]
peptic_ulcer_disease -> <IntegerArray>
[0, 1, <NA>]
Length: 3, dtype: Int64
diabetes -> [1 0]
moderate_to_severe_chronic_kidney_disease -> <IntegerArray>
[0, 1, <NA>]
Length: 3, dtype: Int64
hemiplegia -> [0 1]
leukemia -> [0]
malignant_lymphoma -> [0 1]
solid_tumor -> [0 1]
liver_disease -> <IntegerArray>
[0, 1, <NA>]
Length: 3, dtype: Int64
aids -> [0 1]
type2_respiratory_failure -> [0 1]
acute_renal_failure -> [0 1]


In [37]:
print(df_patienthistory['cci_score'].describe())
print(df_patienthistory['cci_score'].value_counts(dropna=False).sort_index())

count      2003.0
mean     1.861707
std      0.961469
min           0.0
25%           1.0
50%           2.0
75%           2.0
max           6.0
Name: cci_score, dtype: Float64
cci_score
0        56
1       770
2       699
3       368
4        94
5        15
6         1
<NA>      5
Name: count, dtype: Int64


## Checking for Missing values

We check every column for missing values. Most comorbidity flags have none, but `peptic_ulcer_disease`, `moderate_to_severe_chronic_kidney_disease`, `liver_disease`, and `cci_score` each have a small number of missing entries. These are simple binary comorbidity flags with no clear reason for missingness — unlike the hospitalization table, missing here does not reliably mean "confirmed absent." So we leave them as `NaN` rather than filling with 0, which would incorrectly assert the condition was confirmed absent when it may simply not have been recorded.

In [38]:
# Total missing values per column
missing_counts = df_patienthistory.isna().sum()
print(missing_counts)

patient_id                                   0
cerebrovascular_disease                      0
dementia                                     0
chronic_obstructive_pulmonary_disease        0
connective_tissue_disease                    0
peptic_ulcer_disease                         2
diabetes                                     0
moderate_to_severe_chronic_kidney_disease    2
hemiplegia                                   0
leukemia                                     0
malignant_lymphoma                           0
solid_tumor                                  0
liver_disease                                1
aids                                         0
cci_score                                    5
type2_respiratory_failure                    0
acute_renal_failure                          0
dtype: int64


In [39]:
import os

# Create an output folder for cleaned files (keeps them separate from raw data)
output_folder = "cleaned"
os.makedirs(output_folder, exist_ok=True)

# Save each cleaned dataframe
df_demography.to_csv(f"{output_folder}/demography_cleaned.csv", index=False)
df_hospitalization.to_csv(f"{output_folder}/hospitalization_discharge_cleaned.csv", index=False)
df_patienthistory.to_csv(f"{output_folder}/patienthistory_cleaned.csv", index=False)

print("All cleaned files saved to:", output_folder)

All cleaned files saved to: cleaned
